# libCacheSim baselines for twitter_cluster52

Gets LRU/LFU/ARC/LeCaR/LHD/LRB miss ratios at cache sizes 500K/2M/8M, matching the sizes used for the internal NeuraCaR/LRU/LFU/LeCaR comparison. See NOTES.md in the repo for background on the `-t` params (commas, not semicolons, despite `cachesim --help`'s example).

## 1. Dependencies (Colab is Ubuntu with apt/sudo, so this just works)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y cmake libglib2.0-dev liblz4-dev libzstd-dev libboost-all-dev

## 2. Clone the repo (main has everything merged)

In [ ]:
!git clone https://github.com/tengyi-x/neuracar.git
%cd neuracar

## 3. Build LightGBM (needed for the LRB baseline), then libCacheSim with LRB enabled

In [ ]:
!git clone --recursive --depth 1 https://github.com/microsoft/LightGBM.git
!mkdir -p LightGBM/build && cd LightGBM/build && cmake .. -DCMAKE_INSTALL_PREFIX=/usr/local && make -j$(nproc) && make install
!ldconfig

In [ ]:
!bash scripts/setup_libcachesim.sh

In [ ]:
%cd third_party/libCacheSim/_build
!cmake .. -DCMAKE_BUILD_TYPE=Release -DENABLE_LRB=ON
!make -j$(nproc)
%cd /content/neuracar

## 4. Prepare the trace (regenerated from libCacheSim's bundled raw trace, no upload needed)

In [ ]:
!python scripts/prepare_trace.py third_party/libCacheSim/data/twitter_cluster52.csv data/twitter_cluster52_prepared.csv --time-col 0 --obj-id-col 1 --size-col 2 --has-header

## 5. Run all 6 baselines across the 3 cache sizes

In [ ]:
import subprocess, csv, os

CACHESIM = "third_party/libCacheSim/_build/bin/cachesim"
TRACE = "data/twitter_cluster52_prepared.csv"
PARAMS = "time-col=1,obj-id-col=2,size-col=3,delimiter=,,has-header=true"
SIZES = ["500K", "2M", "8M"]
ALGOS = ["LRU", "LFU", "ARC", "LeCaR", "LHD", "LRB"]

env = dict(os.environ)
env["LD_LIBRARY_PATH"] = "/usr/local/lib:" + env.get("LD_LIBRARY_PATH", "")

rows = []
for size in SIZES:
    for algo in ALGOS:
        result = subprocess.run(
            [CACHESIM, TRACE, "csv", algo, size, "-t", PARAMS, "-v", "0"],
            capture_output=True, text=True, env=env,
        )
        output = result.stdout + result.stderr
        miss_ratio = None
        for line in output.splitlines():
            if "miss ratio" in line.lower():
                miss_ratio = float(line.split("miss ratio")[1].strip().split()[0])
        rows.append({"algo": algo, "cache_size": size, "hit_ratio": 1 - miss_ratio if miss_ratio is not None else None})
        print(rows[-1])

with open("twitter_cluster52_libcachesim_baselines.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["algo", "cache_size", "hit_ratio"])
    writer.writeheader()
    writer.writerows(rows)
print("done")

## 6. Download the result

In [ ]:
from google.colab import files
files.download("twitter_cluster52_libcachesim_baselines.csv")